In [ ]:
# Celda 1
import sys; sys.path.append('..')
from src.dataset import get_datasets, visualize_samples
train, test = get_datasets()
visualize_samples(test, n=16)

# Celda 2
import torch
from src.model import CNN_CIFAR10
model = CNN_CIFAR10()
batch = torch.randn(8, 3, 32, 32)
print(model(batch).shape)

# Celda 3 (opcional)
# Celda 3: Entrenamiento Express (Mini-batch) y Visualización de Resultados
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import time

# 1. Configurar un subconjunto de datos para que sea rápido (ej. 5000 imágenes)
from src.dataset import stratified_split
train_subset, _ = stratified_split(train, val_ratio=0.2)

# Tomamos solo las primeras 5000 muestras para este ejemplo rápido
mini_train = torch.utils.data.Subset(train_subset, range(5000))
train_loader = DataLoader(mini_train, batch_size=64, shuffle=True)

# 2. Configurar modelo, pérdida y optimizador
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CNN_CIFAR10().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# 3. Bucle de entrenamiento corto (3 épocas)
EPOCHS = 3
losses = []

print(f"Entrenando en {device}...")
inicio = time.time()

model.train()
for epoch in range(EPOCHS):
    epoch_loss = 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(train_loader)
    losses.append(avg_loss)
    print(f"Época {epoch+1}/{EPOCHS} - Loss: {avg_loss:.4f}")

print(f"Entrenamiento rápido finalizado en {time.time()-inicio:.2f} segundos.")

# 4. Mostrar la gráfica de pérdida (Loss)
plt.figure(figsize=(8, 4))
plt.plot(range(1, EPOCHS+1), losses, marker='o', color='blue', linestyle='-')
plt.title('Curva de Pérdida (Entrenamiento Express)')
plt.xlabel('Época')
plt.ylabel('Loss')
plt.grid(True)
plt.show()

# 5. Mostrar la gráfica oficial generada por train.py
import os
if os.path.exists('../outputs/loss_accuracy.png'):
    print("\n--- Gráfica generada desde src/train.py (Entrenamiento completo) ---")
    img = plt.imread('../outputs/loss_accuracy.png')
    plt.figure(figsize=(12, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.show()
else:
    print("Aún no has ejecutado src/train.py, así que no hay gráfica completa.")